In [31]:
print("hallow world")

hallow world


In [32]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from datetime import datetime, timedelta, time
import os
import os
import sys
import pandas as pd
from typing import List, Dict, Optional
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import warnings

warnings.filterwarnings("ignore")
import plotly.express as px

In [33]:
dataset_path = r'C:\Users\LENOVO\MachineLearningProhects\AutonomousDataAnalystAgent\data\raw\x_data.xlsx'

In [34]:
main_df = pd.read_excel(r'C:\Users\LENOVO\MachineLearningProhects\AutonomousDataAnalystAgent\data\raw\x_data.xlsx')

In [35]:
import pandas as pd

# Original data
d = pd.DataFrame([
    ["BED,BAR", "ATA", 10, 2, 3, 1, 4, 0, 0, 0, 0, 0, 0, 10, 2, 3, 1, 4, 0, 0, 0, 0, 0, 0],
    ["BED,BAR", "BAR-1", 10, 2, 3, 1, 4, 0, 0, 0, 0, 0, 0, 10, 2, 3, 1, 4, 0, 0, 0, 0, 0, 0],
    ["BED,BAR", "BAR-2", 10, 2, 3, 1, 4, 0, 0, 0, 0, 0, 0, 10, 2, 3, 1, 4, 0, 0, 0, 0, 0, 0],
    ["BED,BAR", "BHA", 10, 2, 3, 1, 4, 0, 0, 0, 0, 0, 0, 10, 2, 3, 1, 4, 0, 0, 0, 0, 0, 0],
    ["BED,BAR", "BHE", 10, 2, 3, 1, 4, 0, 0, 0, 0, 0, 0, 10, 2, 3, 1, 4, 0, 0, 0, 0, 0, 0],
], columns=[
    "Division",
    "Sub-Division",
    "Total Complaint Count (A+B+C)",
    "Complain Count (A)",  
    "Hour <2 (A)",
    "Hour 2<4 (A)",
    "Hour 4<8 (A)",
    "Hour <8 (A)",
    "Complain Count (B)",
    "Hour <2 (B)",
    "Hour 2<4 (B)",
    "Hour 4<8 (B)",
    "Hour <8 (B)",
    "Complain Count (C)",
    "Hour <2 (C)",
    "Hour 2<4 (C)",
    "Hour 4<8 (C)",
    "Hour <8 (C)",
    "Closed Out of Total",
    "Hour <2 (Closed)",
    "Hour 2<4 (Closed)",
    "Hour 4<8 (Closed)",
    "Hour <8 (Closed)",    
    "Un-Identified Complaints"
])

# Calculate row-wise total (excluding Division and Sub-Division)
numeric_cols = d.columns[2:]
d['Row Total'] = d[numeric_cols].sum(axis=1)

# Calculate column-wise total
col_totals = d[numeric_cols].sum()
col_totals['Row Total'] = d['Row Total'].sum()

# Build grand total row with correct length
grand_total_values = ['Grand Total', ''] + list(col_totals.values)
grand_total_row = pd.DataFrame([grand_total_values], columns=d.columns)

# Append grand total row
df_final = pd.concat([d, grand_total_row], ignore_index=True)

# Optional: replace zeros with empty strings for display
df_final_display = df_final.replace(0, "")

In [36]:
d

,Division,Sub-Division,Total Complaint Count (A+B+C),Complain Count (A),Hour <2 (A),Hour 2<4 (A),Hour 4<8 (A),Hour <8 (A),Complain Count (B),Hour <2 (B),...,Hour 2<4 (C),Hour 4<8 (C),Hour <8 (C),Closed Out of Total,Hour <2 (Closed),Hour 2<4 (Closed),Hour 4<8 (Closed),Hour <8 (Closed),Un-Identified Complaints,Row Total
0,"BED,BAR",ATA,10,2,3,1,4,0,0,0,...,3,1,4,0,0,0,0,0,0,40
1,"BED,BAR",BAR-1,10,2,3,1,4,0,0,0,...,3,1,4,0,0,0,0,0,0,40
2,"BED,BAR",BAR-2,10,2,3,1,4,0,0,0,...,3,1,4,0,0,0,0,0,0,40
3,"BED,BAR",BHA,10,2,3,1,4,0,0,0,...,3,1,4,0,0,0,0,0,0,40
4,"BED,BAR",BHE,10,2,3,1,4,0,0,0,...,3,1,4,0,0,0,0,0,0,40


In [37]:
def close_power_outage_duration(dataset_path, selected_day):
    # Read dataset
    main_df = pd.read_excel(dataset_path)
    df = main_df.copy()
    
    # Ensure DATE column is datetime
    df['DATE'] = pd.to_datetime(df['DATE'])
    
    # Filter for specific day
    day_filter_data = df[df['DATE'].dt.date == pd.to_datetime(selected_day).date()].copy()
    
    # Normalize complaint type text
    day_filter_data['COMPLAINT TYPE'] = (
        day_filter_data['COMPLAINT TYPE'].astype(str).str.strip().str.title()
    )

    # Filter complaint types
    power_outage_data = day_filter_data[
        day_filter_data['COMPLAINT TYPE'].isin(['Power Outage', 'No Power Supply'])
    ].copy()

    
    # Convert time objects into datetime for subtraction
    def to_datetime(t):
        if pd.isnull(t):
            return pd.NaT
        if isinstance(t, datetime):
            return t
        if isinstance(t, time):
            return datetime.combine(datetime.today(), t)
        return pd.to_datetime(t)
    
    # Apply conversion first
    power_outage_data['COMPLAINT_RECEIVED_DT'] = power_outage_data['COMPLAINT RECEIVED TIME'].apply(to_datetime)
    power_outage_data['FINAL_RESPONSE_DT'] = power_outage_data['FINAL RESPONSE TIME'].apply(to_datetime)
    
    # Fill missing FINAL RESPONSE TIME with current timestamp AFTER conversion
    power_outage_data['FINAL_RESPONSE_DT'] = power_outage_data['FINAL_RESPONSE_DT'].fillna(pd.Timestamp.now())
    
    # Calculate difference in hours
    power_outage_data['DURATION_HOURS'] = (
        power_outage_data['FINAL_RESPONSE_DT'] - power_outage_data['COMPLAINT_RECEIVED_DT']
    ).dt.total_seconds() / 3600
    
    # Round to nearest whole hour
    power_outage_data['DURATION_HOURS_ROUNDED'] = power_outage_data['DURATION_HOURS'].round()
    
    # Integer hours (floor)
    power_outage_data['DURATION_HOURS_INT'] = power_outage_data['DURATION_HOURS'].fillna(0).astype(int)
    
    # Select relevant columns
    close_open_hour = power_outage_data[['DIVISION', 'SUB-DIVISION', 'SHIFT DUTY', 'CLOSED/OPEN', 'DURATION_HOURS_INT']].copy()
    
    # Define classification function
    def classify_duration(x):
        if x < 2:
            return "<2"
        elif 2 <= x < 4:
            return "2<4"
        elif 4 <= x < 8:
            return "4<8"
        else:
            return ">8"

    # Apply classification
    close_open_hour["DURATION_RANGE"] = close_open_hour["DURATION_HOURS_INT"].apply(classify_duration)

    # Pivot table
    pivot = pd.pivot_table(
        close_open_hour,
        values='DURATION_HOURS_INT',
        index=['DIVISION', 'SUB-DIVISION', 'SHIFT DUTY'],
        columns=['DURATION_RANGE', 'CLOSED/OPEN'],
        aggfunc='count',
        fill_value=0,
        margins=True,
        margins_name='Grand Total',
        observed=False
    )

    
    pivot_df = pivot.reset_index()
    pivot_df.columns = ['_'.join([str(c) for c in col if c]) for col in pivot_df.columns.values]

    # Fix: Use the correct column name after pivot (it keeps the space)
    pivot_df_siftA = pivot_df[pivot_df['SHIFT DUTY'] == 'A']
    pivot_df_siftB = pivot_df[pivot_df['SHIFT DUTY'] == 'B']
    pivot_df_siftC = pivot_df[pivot_df['SHIFT DUTY'] == 'C']

    # Concatenate properly
    merge_df = pd.concat([pivot_df_siftA, pivot_df_siftB, pivot_df_siftC], axis=0)
    
    # Sum duration range columns correctly
    duration_cols = [col for col in merge_df.columns if any(dur in col for dur in ['<2_', '2<4_', '4<8_', '>8_'])]
    merge_df["Total Complaint Count (A+B+C)"] = merge_df[duration_cols].sum(axis=1)
    
    # Separate numeric and categorical columns
    numeric_cols = merge_df.select_dtypes(include='number').columns
    categorical_cols = merge_df.select_dtypes(exclude='number').columns

    # Create totals for numeric columns
    totals = merge_df[numeric_cols].sum().astype(int)

    # Fill categorical columns with a label
    for col in categorical_cols:
        totals[col] = "Grand Total"

    # Append totals row
    merge_df.loc['Grand Total'] = totals
    
    # Select final columns
    final_cols = ['DIVISION', 'SUB-DIVISION', 'SHIFT DUTY', 'Total Complaint Count (A+B+C)']
    final_cols.extend([col for col in merge_df.columns if 'Grand Total' in col or col in duration_cols])
    
    final_df = merge_df[final_cols]
    
    return final_df

In [41]:
selected_day = "2025-07-29"

In [42]:
pvt = close_power_outage_duration(dataset_path, selected_day)

In [43]:
pvt

,DIVISION,SUB-DIVISION,SHIFT DUTY,Total Complaint Count (A+B+C),2<4_Closed,4<8_Closed,<2_Closed,Grand Total
0,BARGARH,BARGARH-2,A,1,0,1,0,1
1,BOLANGIR,BOLANGIR-2,A,1,0,0,1,1
2,KEED,KESINGA,A,3,0,2,1,3
4,KWED,DHARAMGARH,A,2,0,2,0,2
5,NUAPADA,KHARIAR,A,3,0,1,2,3
6,SAMBALPUR,"SDO-1, AINTHAPALI",A,1,0,0,1,1
7,SUNDERGARH,SUNDERGARH,A,1,0,0,1,1
9,TITLAGARH,KANTABANJI,A,2,0,0,2,2
10,TITLAGARH,KANTABANJI,B,2,1,0,1,2
3,KEED,KESINGA,C,3,0,2,1,3
